# Training a Model on Encrypted Data

In [ ]:
import sys
import torch
import torchvision
import matplotlib.pyplot as plt


# python 3.7 is required
assert sys.version_info[0] == 3 and sys.version_info[1] == 7, "python 3.7 is required"

import shardwise
shardwise.init()

%matplotlib inline

## MNIST

In [ ]:
digits = torchvision.datasets.MNIST(root='/tmp/data', 
                                           train=True, 
                                           transform=torchvision.transforms.ToTensor(),
                                           download=True)

digits_test = torchvision.datasets.MNIST(root='/tmp/data', 
                                           train=True, 
                                           transform=torchvision.transforms.ToTensor(),
                                           download=True)

In [ ]:
plt.imshow(digits[0][0][0], cmap='gray', interpolation='none')

In [ ]:
print("label for image is ", digits[0][1])

## Preprocess Into Tensors

In [ ]:
def take_samples(digits, n_samples=1000):
    """Returns images and labels based on sample size"""
    images, labels = [], []

    for i, digit in enumerate(digits):
        if i == n_samples:
            break
        image, label = digit
        images.append(image)
        label_one_hot = torch.nn.functional.one_hot(torch.tensor(label), 10)
        labels.append(label_one_hot)

    images = torch.cat(images)
    labels = torch.stack(labels)
    return images, labels
    

In [ ]:
images, labels = take_samples(digits, n_samples=100)

In [ ]:
print(images.shape)
print(labels.shape)

In [ ]:
images_enc = shardwise.tensor(images)
labels_enc = shardwise.tensor(labels)

In [ ]:
images_enc[0]

In [ ]:
# test set
images_test, labels_test = take_samples(digits_test, n_samples=20)
images_test_enc = shardwise.tensor(images_test)
labels_test_enc = shardwise.tensor(labels_test)

## Logistic Regression Model

(multiclass logistic regression)

In [ ]:
class LogisticRegression(shardwise.nn.Module):
    
    def __init__(self):
        super().__init__()
        # images are 28x28 pixels
        self.linear = shardwise.nn.Linear(28 * 28, 10)
        
    def forward(self, x):
        return self.linear(x)
        
    

In [ ]:
model = LogisticRegression().encrypt()

In [ ]:
model(images_enc)

### Train Model on Encrypted Images

In [ ]:
def train_model(model, X, y, epochs=10, learning_rate=0.05):
    criterion = shardwise.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        print(f"epoch {epoch} loss: {loss.get_plain_text()}")
        loss.backward()
        model.update_parameters(learning_rate)
    return model

In [ ]:
model = train_model(model, images_enc, labels_enc)

### Decrypt Prediction

In [ ]:
prediction = model(images_enc[3].unsqueeze(0))

In [ ]:
prediction.get_plain_text().argmax()

In [ ]:
plt.imshow(images[3], cmap='gray', interpolation='none')

### Test Model Accuracy

In [ ]:
def avg_test_accuracy(model, X, y):    
    output = model(X).get_plain_text().softmax(0)
    predicted = output.argmax(1)
    labels = y.get_plain_text().argmax(1)
    correct = (predicted == labels).sum().float()
    return float(correct / y.shape[0])

In [ ]:
avg_test_accuracy(model, images_enc, labels_enc)

## Training a CNN

based on https://github.com/pytorch/examples/blob/master/mnist/main.py

In [ ]:
class CNN(shardwise.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = shardwise.nn.Conv2d(1, 32, 3, 1)
        self.conv2 = shardwise.nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = shardwise.nn.Dropout2d(0.25)
        self.dropout2 = shardwise.nn.Dropout2d(0.5)
        self.fc1 = shardwise.nn.Linear(9216, 128)
        self.fc2 = shardwise.nn.Linear(128, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = x.relu()
        x = self.conv2(x)
        x = x.relu()
        x = x.max_pool2d(2)
        x = self.dropout1(x)
        x = self.fc1(x)
        x = x.relu()
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

In [ ]:
model = CNN().encrypt()

In [ ]:
x = images_enc[0].unsqueeze(0)
print(x.shape)
model(x)

In [ ]:
model = train_model(model, images_enc[:10, ], labels_enc[:10,], epochs=3)

In [ ]:
prediction = model(images_enc[3].unsqueeze(0)).argmax()

# Importing PyTorch Models

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class PyTorchModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout2d(0.25)
        self.dropout2 = nn.Dropout2d(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [ ]:
pytorch_model = PyTorchModel()

In [ ]:
dummy_input = torch.empty(images.shape)
shardwise_model = shardwise.nn.from_pytorch(pytorch_model, dummy_input)
shardwise_model.encrypt()

In [ ]:
prediction = shardwise_model(images_enc[3].unsqueeze(0))
print(prediction)

In [ ]:
print(prediction.get_plain_text())

In [ ]:
prediction.get_plain_text().argmax()

In [ ]:
shardwise_model = train_model(shardwise_model, images_enc[:10, ], labels_enc[:10,], epochs=3)